# Module 7 Instructor Colab Notebook: Sequences, Series, and Taylor Expansions

## A Mathematical Bridge to Artificial Intelligence

AI algorithms are iterative: they create a sequence of parameter values, losses, predictions, or hidden states. Understanding whether such a sequence settles down requires the language of convergence. Infinite series then describe accumulated contributions, and Taylor expansions replace complicated functions by local polynomials.

These ideas meet directly in optimization. A derivative-based update trusts a local approximation. A small step tends to keep that approximation reliable; an oversized step can move beyond the region where the approximation describes the loss and can cause oscillation or divergence.

**Recommended pacing:** three class meetings. The convergence proofs and remainder bounds can be instructor-led extensions while retaining the conceptual storyline.


> **How to use this notebook in class**
>
> Ask students to distinguish terms from partial sums before discussing convergence. When plotting Taylor polynomials, emphasize that agreement is local and depends on both degree and distance from the expansion center. In the optimization section, have students predict stable and unstable trajectories from the error multiplier before running the visualization.

## Notebook map

| Section | Main mathematical ideas | Direct AI connection |
|---|---|---|
| 1 | Sequences and limits | Parameter and loss trajectories |
| 2 | Infinite series and partial sums | Discounted returns and accumulated effects |
| 3 | Taylor polynomials and Taylor series | Polynomial approximations to nonlinear operations |
| 4 | First- and second-order approximations | Slope, curvature, and local loss models |
| 5 | Learning-rate stability | Overshooting, oscillation, and divergence |
| 6 | Approximation error and computation | Truncating expensive nonlinear functions |
| Integrated lab | Optimize an exponential loss | Compare Taylor, gradient, and Newton ideas |


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
# Reusable instructor-solution helper.
def reveal_markdown(button_text, solution_markdown):
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


def tangent_line(function, derivative, point, x_values):
    return function(point) + derivative(point) * (x_values - point)


## Module learning objectives

By the end of the module, students should be able to:

1. represent a sequence explicitly or recursively;
2. interpret and verify basic sequence limits;
3. distinguish a sequence of terms from a sequence of partial sums;
4. analyze geometric and harmonic series;
5. construct Taylor polynomials from derivatives;
6. use first- and second-order approximations;
7. quantify Taylor truncation error in basic cases;
8. derive a stability interval for gradient descent on a quadratic; and
9. explain why curvature changes the safe learning-rate scale.

## Prerequisite checkpoint

Students should know limits, derivatives, the chain rule, and basic integration from Modules 5 and 6. Familiarity with exponentials and logarithms is assumed.


---

# Section 1: Infinite Sequences and Their Limits

## 1. What is a sequence?

A **sequence** is an ordered list of numbers. Formally, it is a function whose input is an integer index:

$$
a_1,a_2,a_3,\ldots
$$

The symbol $$a_n$$ denotes the term at index $$n.$$ An **explicit formula** computes a term directly from its index, such as

$$
a_n=\frac{1}{n}.
$$

A **recursive formula** computes a new term from earlier terms, such as

$$
a_{n+1}=\frac{1}{2}a_n,
\qquad
a_0=8.
$$

The recursive sequence begins

$$
8,4,2,1,\frac{1}{2},\ldots
$$


## 2. Convergence and divergence

A sequence $$a_n$$ **converges** to $$L$$ if its terms become arbitrarily close to $$L$$ for all sufficiently large indices. We write

$$
\lim_{n\to\infty}a_n=L.
$$

Formally, for every $$\varepsilon>0,$$ there is an integer $$N$$ such that

$$
n\geq N
\quad\Longrightarrow\quad
|a_n-L|<\varepsilon.
$$

A sequence that has no finite limit **diverges**. Divergence can occur because terms grow without bound, oscillate without settling, or behave irregularly.


## 3. Detailed convergence proof for a reciprocal sequence

Consider

$$
a_n=\frac{1}{n}.
$$

We claim

$$
\lim_{n\to\infty}\frac{1}{n}=0.
$$

Given $$\varepsilon>0,$$ we need

$$
\left|\frac{1}{n}-0\right|<\varepsilon.
$$

This is equivalent to

$$
\frac{1}{n}<\varepsilon,
$$

which holds when

$$
n>\frac{1}{\varepsilon}.
$$

Choose any integer $$N$$ satisfying

$$
N>\frac{1}{\varepsilon}.
$$

Then for every $$n\geq N,$$

$$
0<\frac{1}{n}\leq \frac{1}{N}<\varepsilon.
$$

Therefore the sequence converges to zero.


In [ ]:
n_sequence = np.arange(1, 101)
reciprocal_sequence = 1/n_sequence
oscillating_sequence = (-1.0)**n_sequence

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(n_sequence, reciprocal_sequence, s=18)
axes[0].axhline(0, color="crimson", linestyle="--")
axes[0].set_xlabel("index n")
axes[0].set_ylabel("term")
axes[0].set_title("The sequence one over n converges to zero")
axes[1].scatter(n_sequence[:30], oscillating_sequence[:30], s=25)
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set_xlabel("index n")
axes[1].set_ylabel("term")
axes[1].set_title("Alternating signs need not settle")
plt.tight_layout()
plt.show()


## 4. Common sequence behaviors

| Sequence | Behavior | Reason |
|---|---|---|
| $$a_n=1/n$$ | Converges to zero | Denominator grows without bound |
| $$a_n=3+2/n$$ | Converges to three | Added term approaches zero |
| $$a_n=(-1)^n$$ | Diverges | Alternates between negative one and one |
| $$a_n=n^2$$ | Diverges to infinity | Terms grow without bound |
| $$a_n=r^n$$ with $$|r|<1$$ | Converges to zero | Repeated multiplication shrinks magnitude |

A convergent sequence must be bounded, but a bounded sequence need not converge. The sequence $$(-1)^n$$ is bounded yet divergent.


## 5. Iterative algorithms generate sequences

Gradient descent creates the recursive sequence

$$
w_{k+1}=w_k-\eta L'(w_k).
$$

For

$$
L(w)=(w-3)^2,
$$

we derived

$$
w_{k+1}-3=(1-2\eta)(w_k-3).
$$

Let the error be

$$
e_k=w_k-3.
$$

Then

$$
e_{k+1}=(1-2\eta)e_k.
$$

Repeated substitution gives

$$
e_k=(1-2\eta)^ke_0.
$$

Thus the training sequence converges to the minimizer when

$$
|1-2\eta|<1.
$$

This is a geometric-sequence condition.


> **Common mistakes in Section 1**
>
> - Treating the index as a continuous variable without explanation.
> - Checking only the first few terms and declaring a limit.
> - Assuming boundedness guarantees convergence.
> - Confusing oscillation with convergence to the average of oscillating values.
> - Confusing a recursive update with a closed-form term.

## Section 1 terminology

| Term | Meaning |
|---|---|
| Sequence | Function defined on integer indices |
| Term | One value in a sequence |
| Index | Integer position identifying a term |
| Explicit formula | Direct expression for the indexed term |
| Recursive formula | Rule producing a term from preceding terms |
| Convergent sequence | Sequence approaching a finite limit |
| Divergent sequence | Sequence with no finite limit |
| Bounded sequence | Sequence whose terms stay within fixed finite bounds |
| Training trajectory | Sequence of parameters or losses produced during training |


## Section 1 guided worksheet

1. Write the first five terms of

   $$
   a_n=2+\left(\frac{1}{3}\right)^n,
   \qquad n=0,1,2,\ldots
   $$

2. Find its limit.
3. Solve the recurrence

   $$
   e_{k+1}=-0.6e_k,
   \qquad e_0=5,
   $$

   and describe its behavior.
4. Give an example showing that a bounded sequence can diverge.


In [ ]:
reveal_markdown("Open Section 1 solutions", r"""
1. Substitute consecutive indices:

   $$
   a_0=2+1=3,
   $$

   $$
   a_1=2+\frac{1}{3}=\frac{7}{3},
   $$

   $$
   a_2=2+\frac{1}{9}=\frac{19}{9},
   $$

   $$
   a_3=2+\frac{1}{27}=\frac{55}{27},
   $$

   and

   $$
   a_4=2+\frac{1}{81}=\frac{163}{81}.
   $$

2. Since

   $$
   \left(\frac{1}{3}\right)^n\to 0,
   $$

   we have

   $$
   \boxed{a_n\to 2}.
   $$

3. Repeated substitution gives

   $$
   \boxed{e_k=5(-0.6)^k}.
   $$

   Its sign alternates, but its magnitude is

   $$
   5(0.6)^k\to 0.
   $$

   Thus it converges to zero by shrinking oscillation.

4. The sequence $$(-1)^n$$ stays between negative one and one but alternates forever, so it is bounded and divergent.
""")


---

# Section 2: Infinite Series and Partial Sums

## 1. A series is not merely a sequence of terms

An **infinite series** is a formal sum

$$
\sum_{n=0}^{\infty}a_n
=a_0+a_1+a_2+\cdots.
$$

To define its value, form the **partial sums**

$$
S_N=\sum_{n=0}^{N}a_n.
$$

The series converges to $$S$$ when the sequence of partial sums converges:

$$
\lim_{N\to\infty}S_N=S.
$$

If the partial sums do not approach a finite limit, the series diverges.


## 2. Necessary term test

If

$$
\sum_{n=0}^{\infty}a_n
$$

converges, then necessarily

$$
a_n\to 0.
$$

To see this, note that

$$
a_N=S_N-S_{N-1}.
$$

If both partial sums approach the same limit $$S,$$ then

$$
a_N\to S-S=0.
$$

The converse is false: terms approaching zero do not guarantee that their accumulated sum is finite.


## 3. Geometric series: full derivation

A geometric series has the form

$$
\sum_{n=0}^{\infty}ar^n
=a+ar+ar^2+\cdots.
$$

Its finite partial sum is

$$
S_N=a+ar+ar^2+\cdots+ar^N.
$$

Multiply by $$r$$:

$$
rS_N=ar+ar^2+\cdots+ar^N+ar^{N+1}.
$$

Subtract the second equation from the first:

$$
S_N-rS_N=a-ar^{N+1}.
$$

Factor:

$$
(1-r)S_N=a(1-r^{N+1}).
$$

For $$r\ne 1,$$

$$
S_N=a\frac{1-r^{N+1}}{1-r}.
$$

If $$|r|<1,$$ then $$r^{N+1}\to 0,$$ so

$$
\boxed{\sum_{n=0}^{\infty}ar^n=\frac{a}{1-r}.}
$$

If $$|r|\geq 1,$$ the terms fail to approach zero or the partial sums fail to settle, so the series diverges.


In [ ]:
ratios_series = [0.5, 0.9, -0.7, 1.05]
n_partial = np.arange(0, 31)
fig, ax = plt.subplots()
for ratio in ratios_series:
    terms = ratio**n_partial
    partial = np.cumsum(terms)
    ax.plot(n_partial, partial, marker="o", markersize=3, label=f"r = {ratio}")
ax.set_xlabel("largest included index N")
ax.set_ylabel("partial sum")
ax.set_title("A series converges when its partial sums settle")
ax.legend()
plt.show()


## 4. Harmonic series: terms vanish but the sum diverges

The harmonic series is

$$
\sum_{n=1}^{\infty}\frac{1}{n}.
$$

Its terms approach zero, but group them as follows:

$$
1+\frac{1}{2}
+\left(\frac{1}{3}+\frac{1}{4}\right)
+\left(\frac{1}{5}+\cdots+\frac{1}{8}\right)
+\cdots.
$$

In the third group, each term is at least $$1/4,$$ so the group sum is at least

$$
2\left(\frac{1}{4}\right)=\frac{1}{2}.
$$

In the next group, each term is at least $$1/8,$$ and there are four terms, so its sum is at least

$$
4\left(\frac{1}{8}\right)=\frac{1}{2}.
$$

Every subsequent doubled group contributes at least one-half. Therefore the partial sums eventually exceed any finite bound, and

$$
\boxed{\sum_{n=1}^{\infty}\frac{1}{n}\text{ diverges}.}
$$


## 5. The p-series benchmark

The series

$$
\sum_{n=1}^{\infty}\frac{1}{n^p}
$$

converges exactly when

$$
p>1.
$$

It diverges for $$p\leq 1.$$ This mirrors the improper-integral benchmark from Module 6.

For example,

$$
\sum_{n=1}^{\infty}\frac{1}{n^2}
$$

converges, while

$$
\sum_{n=1}^{\infty}\frac{1}{n}
$$

diverges.


## 6. Direct AI application: discounted return

In reinforcement learning, a **discounted return** can be written

$$
G_t=r_t+\gamma r_{t+1}+\gamma^2r_{t+2}+\cdots,
$$

where $$0\leq\gamma<1$$ is the **discount factor**. It makes later rewards contribute less strongly.

If every reward equals a constant $$r,$$ then

$$
G=r+r\gamma+r\gamma^2+\cdots.
$$

This is a geometric series with first term $$r$$ and ratio $$\gamma,$$ so

$$
\boxed{G=\frac{r}{1-\gamma}.}
$$

For $$r=2$$ and $$\gamma=0.9,$$

$$
G=\frac{2}{1-0.9}=20.
$$

After retaining rewards through index $$N,$$ the omitted tail is

$$
\frac{r\gamma^{N+1}}{1-\gamma}.
$$


In [ ]:
gamma_return = 0.9
reward_return = 2.0
n_return = np.arange(0, 61)
partial_return = reward_return*(1-gamma_return**(n_return+1))/(1-gamma_return)
exact_return = reward_return/(1-gamma_return)

fig, ax = plt.subplots()
ax.plot(n_return, partial_return, linewidth=2)
ax.axhline(exact_return, color="crimson", linestyle="--", label="infinite return")
ax.set_xlabel("last included reward index")
ax.set_ylabel("discounted partial return")
ax.set_title("Discounted rewards form a convergent geometric series")
ax.legend()
plt.show()


> **Common mistakes in Section 2**
>
> - Checking convergence of the terms instead of the partial sums.
> - Believing that $$a_n\to 0$$ guarantees series convergence.
> - Using the infinite geometric formula when the ratio magnitude is at least one.
> - Losing the first term or index shift in a geometric series.
> - Treating a truncated sum as exact without accounting for its tail.

## Section 2 terminology

| Term | Meaning |
|---|---|
| Infinite series | Limit-based sum of infinitely many terms |
| Partial sum | Finite sum through a specified index |
| Geometric series | Series whose consecutive terms have constant ratio |
| Harmonic series | Divergent series of reciprocal positive integers |
| p-series | Reciprocal-power series controlled by exponent $$p$$ |
| Discount factor | Multiplier reducing the influence of future rewards |
| Truncation tail | Omitted terms after a finite cutoff |


## Section 2 guided worksheet

1. Determine whether the geometric series

   $$
   5-2.5+1.25-0.625+\cdots
   $$

   converges, and find its sum.

2. Find the first four partial sums of

   $$
   \sum_{n=1}^{\infty}\frac{1}{n}.
   $$

3. A constant reward is $$3$$ and the discount factor is $$0.8.$$ Find the infinite discounted return.
4. Find the tail after retaining terms through $$n=4.$$


In [ ]:
reveal_markdown("Open Section 2 solutions", r"""
1. The first term is $$a=5$$ and the ratio is

   $$
   r=\frac{-2.5}{5}=-0.5.
   $$

   Since $$|r|<1,$$ the series converges to

   $$
   \frac{5}{1-(-0.5)}
   =\frac{5}{1.5}
   =\boxed{\frac{10}{3}}.
   $$

2. The first four partial sums are

   $$
   S_1=1,
   $$

   $$
   S_2=1+\frac{1}{2}=\frac{3}{2},
   $$

   $$
   S_3=\frac{3}{2}+\frac{1}{3}=\frac{11}{6},
   $$

   and

   $$
   S_4=\frac{11}{6}+\frac{1}{4}=\frac{25}{12}.
   $$

3. The infinite return is

   $$
   \frac{3}{1-0.8}=\boxed{15}.
   $$

4. Retaining indices zero through four omits the tail beginning with index five:

   $$
   \frac{3(0.8)^5}{1-0.8}
   =15(0.8)^5
   =\boxed{4.9152}.
   $$
""")


---

# Section 3: Taylor Polynomials and Taylor Series

## 1. Why approximate by a polynomial?

Polynomials are easy to evaluate, differentiate, integrate, and optimize. A **Taylor polynomial** approximates a smooth function near a chosen **center** by matching derivative information at that point.

For a center $$a,$$ the degree-$$n$$ Taylor polynomial is

$$
\boxed{T_n(x)=\sum_{k=0}^{n}\frac{f^{(k)}(a)}{k!}(x-a)^k.}
$$

Here:

- $$f^{(k)}$$ is the $$k$$th derivative;
- $$k!$$ is the factorial;
- $$0!=1$$; and
- the polynomial matches the first $$n$$ derivatives of $$f$$ at $$a.$$


## 2. Derive the coefficient formula by matching derivatives

Begin with a polynomial centered at $$a$$:

$$
P_n(x)=c_0+c_1(x-a)+c_2(x-a)^2+\cdots+c_n(x-a)^n.
$$

At $$x=a,$$

$$
P_n(a)=c_0,
$$

so matching $$f(a)$$ requires

$$
c_0=f(a).
$$

Differentiate and evaluate at $$a$$:

$$
P_n'(a)=c_1,
$$

so

$$
c_1=f'(a).
$$

Differentiate twice:

$$
P_n''(a)=2c_2,
$$

so

$$
c_2=\frac{f''(a)}{2!}.
$$

In general,

$$
P_n^{(k)}(a)=k!c_k.
$$

Matching $$f^{(k)}(a)$$ gives

$$
c_k=\frac{f^{(k)}(a)}{k!},
$$

which produces the Taylor formula.


## 3. Maclaurin series for the exponential

A Taylor expansion centered at zero is called a **Maclaurin expansion**. For

$$
f(x)=e^x,
$$

every derivative is $$e^x,$$ so

$$
f^{(k)}(0)=1
$$

for every $$k.$$ Therefore

$$
T_n(x)=1+x+\frac{x^2}{2!}+\frac{x^3}{3!}+\cdots+\frac{x^n}{n!}.
$$

The corresponding infinite Taylor series is

$$
\boxed{e^x=\sum_{k=0}^{\infty}\frac{x^k}{k!}}
$$

for every real $$x.$$

The factorial grows quickly, allowing the series to converge even when $$|x|>1.$$


In [ ]:
def exp_taylor(x, degree):
    x_array = np.asarray(x, dtype=float)
    result = np.zeros_like(x_array)
    term = np.ones_like(x_array)
    result += term
    for k in range(1, degree+1):
        term = term*x_array/k
        result += term
    return result

x_taylor_exp = np.linspace(-3, 3, 600)
fig, ax = plt.subplots()
ax.plot(x_taylor_exp, np.exp(x_taylor_exp), color="black", linewidth=3,
        label="exponential")
for degree in [1, 2, 4, 8]:
    ax.plot(x_taylor_exp, exp_taylor(x_taylor_exp, degree),
            label=f"degree {degree}")
ax.set_ylim(-2, 22)
ax.set_xlabel("x")
ax.set_ylabel("value")
ax.set_title("Higher degree extends the useful approximation region")
ax.legend()
plt.show()


## 4. Taylor series, convergence, and analyticity

The formal Taylor series centered at $$a$$ is

$$
\sum_{k=0}^{\infty}\frac{f^{(k)}(a)}{k!}(x-a)^k.
$$

Three separate questions matter:

1. Does the numerical series converge?
2. For which values of $$x$$ does it converge?
3. Where it converges, does its sum equal the original function?

The **radius of convergence** is a number $$R$$ such that the series converges when

$$
|x-a|<R
$$

and diverges when

$$
|x-a|>R.
$$

A function equal to its Taylor series near the center is called **analytic** there. Smoothness alone does not guarantee this equality, an important conceptual caution.


## 5. Sine and logarithm examples

The Maclaurin series for sine is

$$
\sin x=x-\frac{x^3}{3!}+\frac{x^5}{5!}-\frac{x^7}{7!}+\cdots.
$$

Only odd powers appear because the even derivatives vanish at zero.

For $$|x|<1,$$

$$
\ln(1+x)
=x-\frac{x^2}{2}+\frac{x^3}{3}-\frac{x^4}{4}+\cdots.
$$

Unlike the exponential series, the logarithm series has a finite radius of convergence centered at zero. A polynomial degree that works well near zero may be unreliable far from zero.


## 6. Taylor remainder

The **remainder** after degree $$n$$ is

$$
R_n(x)=f(x)-T_n(x).
$$

Under standard smoothness conditions, the Lagrange remainder has the form

$$
R_n(x)=\frac{f^{(n+1)}(\xi)}{(n+1)!}(x-a)^{n+1}
$$

for some $$\xi$$ between $$a$$ and $$x.$$

If

$$
|f^{(n+1)}(t)|\leq M
$$

throughout that interval, then

$$
\boxed{|R_n(x)|\leq \frac{M}{(n+1)!}|x-a|^{n+1}.}
$$

The error usually improves when the degree increases or when $$x$$ is closer to the center.


### Certified numerical remainder example

Approximate $$e^{0.5}$$ with the degree-two Maclaurin polynomial

$$
T_2(x)=1+x+\frac{x^2}{2}.
$$

On $$[0,0.5],$$ the third derivative is $$e^x,$$ so it is bounded by

$$
M=e^{0.5}.
$$

The Lagrange bound is

$$
|R_2(0.5)|\leq \frac{e^{0.5}(0.5)^3}{3!}\approx 0.03435.
$$

The actual error is

$$
e^{0.5}-\left(1+0.5+\frac{0.5^2}{2}\right)\approx 0.02372,
$$

which is below the certified bound. A bound is a guarantee, not a prediction that equality will occur.


> **Common mistakes in Section 3**
>
> - Forgetting the factorial in Taylor coefficients.
> - Using powers of $$x$$ when the center is not zero.
> - Assuming a Taylor polynomial is exact everywhere.
> - Assuming convergence of the series automatically means equality with the function.
> - Ignoring the distance from the expansion center.
> - Treating a plotted visual match as a proof of an error bound.

## Section 3 terminology

| Term | Meaning |
|---|---|
| Taylor polynomial | Finite polynomial matching derivatives at a center |
| Taylor series | Infinite series formed from all derivative coefficients |
| Maclaurin expansion | Taylor expansion centered at zero |
| Factorial | Product of positive integers through the index, with zero factorial equal to one |
| Center | Point where derivatives are matched |
| Radius of convergence | Distance range in which a power series converges |
| Analytic function | Function equal to its Taylor series locally |
| Remainder | Difference between the function and truncated polynomial |


## Section 3 guided worksheet

1. Construct the degree-three Maclaurin polynomial for $$e^x.$$
2. Use it to approximate $$e^{0.2}.$$
3. Construct the degree-three Maclaurin polynomial for $$\sin x.$$
4. Explain why a degree-four polynomial centered at zero may approximate a function well at $$x=0.1$$ but poorly at $$x=10.$$


In [ ]:
reveal_markdown("Open Section 3 solutions", r"""
1. The degree-three polynomial is

   $$
   \boxed{T_3(x)=1+x+\frac{x^2}{2}+\frac{x^3}{6}}.
   $$

2. Substitute $$x=0.2$$:

   $$
   T_3(0.2)
   =1+0.2+\frac{0.2^2}{2}+\frac{0.2^3}{6}.
   $$

   Calculate each contribution:

   $$
   \frac{0.2^2}{2}=0.02,
   $$

   $$
   \frac{0.2^3}{6}=\frac{0.008}{6}\approx 0.001333.
   $$

   Therefore

   $$
   T_3(0.2)\approx \boxed{1.221333}.
   $$

3. The degree-three sine polynomial is

   $$
   \boxed{T_3(x)=x-\frac{x^3}{6}}.
   $$

4. Taylor coefficients match local derivative information at the center. The omitted remainder contains a power of the distance from that center. The distance $$0.1$$ is small, whereas the distance $$10$$ can magnify omitted terms enormously.
""")


---

# Section 4: First- and Second-Order Taylor Approximations

## 1. First-order approximation

The first-order Taylor polynomial at $$a$$ is

$$
T_1(x)=f(a)+f'(a)(x-a).
$$

This is the tangent-line or **linear approximation**. Writing a step as

$$
h=x-a,
$$

gives

$$
\boxed{f(a+h)\approx f(a)+f'(a)h.}
$$

It uses the current value and slope while ignoring curvature and higher-order effects.


## 2. Detailed linearization example

Approximate

$$
\sqrt{4.1}
$$

using

$$
f(x)=\sqrt{x}
$$

near $$a=4.$$ We have

$$
f(4)=2
$$

and

$$
f'(x)=\frac{1}{2\sqrt{x}},
$$

so

$$
f'(4)=\frac{1}{4}.
$$

The step is

$$
h=4.1-4=0.1.
$$

Thus

$$
\sqrt{4.1}
\approx 2+\frac{1}{4}(0.1)
=\boxed{2.025}.
$$

The actual value is approximately $$2.024846,$$ so the local linear approximation is accurate here.


## 3. Second-order approximation and curvature

The second-order Taylor polynomial is

$$
T_2(x)=f(a)+f'(a)(x-a)+\frac{1}{2}f''(a)(x-a)^2.
$$

Using a step $$h,$$

$$
\boxed{f(a+h)\approx f(a)+f'(a)h+\frac{1}{2}f''(a)h^2.}
$$

The second derivative adds curvature. If $$f''(a)>0,$$ the quadratic correction bends upward. If $$f''(a)<0,$$ it bends downward.


## 4. First versus second order for the exponential

At $$a=0,$$

$$
e^0=1,
\qquad
\frac{d}{dx}e^x\bigg|_{x=0}=1,
\qquad
\frac{d^2}{dx^2}e^x\bigg|_{x=0}=1.
$$

Therefore

$$
T_1(x)=1+x
$$

and

$$
T_2(x)=1+x+\frac{x^2}{2}.
$$

At $$x=0.5,$$

$$
T_1(0.5)=1.5,
$$

$$
T_2(0.5)=1+0.5+\frac{0.25}{2}=1.625,
$$

while

$$
e^{0.5}\approx 1.648721.
$$

The curvature correction substantially reduces the error.


In [ ]:
h_local = np.linspace(-1.5, 1.5, 500)
true_local = np.exp(h_local)
first_local = 1+h_local
second_local = 1+h_local+0.5*h_local**2

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(h_local, true_local, color="black", linewidth=2, label="true function")
axes[0].plot(h_local, first_local, linestyle="--", label="first order")
axes[0].plot(h_local, second_local, linestyle=":", linewidth=2, label="second order")
axes[0].set_xlabel("step from center")
axes[0].set_ylabel("value")
axes[0].set_title("Local polynomial models")
axes[0].legend()
axes[1].semilogy(h_local, np.maximum(np.abs(true_local-first_local), 1e-15),
                  label="first-order error")
axes[1].semilogy(h_local, np.maximum(np.abs(true_local-second_local), 1e-15),
                  label="second-order error")
axes[1].set_xlabel("step h from the center")
axes[1].set_ylabel("absolute error")
axes[1].set_title("Approximation error grows away from the center")
axes[1].legend()
plt.tight_layout()
plt.show()


## 5. Local loss models

For a one-variable loss and a parameter update $$h,$$ first order gives

$$
L(w+h)\approx L(w)+L'(w)h.
$$

Second order gives

$$
L(w+h)\approx L(w)+L'(w)h+\frac{1}{2}L''(w)h^2.
$$

The first derivative identifies downhill direction. The second derivative estimates how rapidly the slope changes. High positive curvature means the loss can bend upward quickly, so a step that looks safe under the tangent line can overshoot.

In several variables, Module 8 will replace the derivative by a gradient and the second derivative by a Hessian matrix.


> **Common mistakes in Section 4**
>
> - Omitting the center shift $$x-a.$$
> - Treating the approximation symbol as exact equality.
> - Forgetting the factor one-half in the quadratic term.
> - Interpreting curvature as slope.
> - Assuming second order is always accurate for a large step.

## Section 4 terminology

| Term | Meaning |
|---|---|
| Linearization | First-order tangent approximation |
| First-order approximation | Local model using value and first derivative |
| Second-order approximation | Local model adding second-derivative curvature |
| Curvature | Rate at which slope changes |
| Expansion center | Point where derivative information is evaluated |
| Step | Displacement from the expansion center |
| Local model | Simplified approximation intended near a selected point |


## Section 4 guided worksheet

Let

$$
f(x)=\ln x
$$

and center at $$a=1.$$

1. Compute $$f(1),$$ $$f'(1),$$ and $$f''(1).$$
2. Construct first- and second-order Taylor approximations.
3. Use both to approximate $$\ln(1.1).$$
4. Compare the two approximations with the numerical value $$0.095310.$$


In [ ]:
reveal_markdown("Open Section 4 solutions", r"""
1. We have

   $$
   f(1)=\ln 1=0,
   $$

   $$
   f'(x)=\frac{1}{x}
   \quad\Longrightarrow\quad
   f'(1)=1,
   $$

   and

   $$
   f''(x)=-\frac{1}{x^2}
   \quad\Longrightarrow\quad
   f''(1)=-1.
   $$

2. The first-order approximation is

   $$
   T_1(x)=x-1.
   $$

   The second-order approximation is

   $$
   T_2(x)=(x-1)-\frac{1}{2}(x-1)^2.
   $$

3. At $$x=1.1,$$ the step is $$0.1.$$ Thus

   $$
   T_1(1.1)=0.1,
   $$

   while

   $$
   T_2(1.1)=0.1-\frac{1}{2}(0.1)^2
   =0.1-0.005
   =0.095.
   $$

4. The errors are approximately

   $$
   |0.095310-0.1|=0.004690
   $$

   and

   $$
   |0.095310-0.095|=0.000310.
   $$

   The second-order approximation is more accurate for this step.
""")


---

# Section 5: Taylor Reasoning and Learning-Rate Stability

## 1. Insert a gradient step into the local loss model

Gradient descent chooses

$$
h=-\eta L'(w).
$$

Substitute this into the second-order approximation:

$$
L(w+h)
\approx L(w)+L'(w)(-\eta L'(w))
+\frac{1}{2}L''(w)(-\eta L'(w))^2.
$$

Simplify:

$$
L(w+h)-L(w)
\approx -\eta\bigl(L'(w)\bigr)^2
+\frac{1}{2}\eta^2L''(w)\bigl(L'(w)\bigr)^2.
$$

Factor:

$$
L(w+h)-L(w)
\approx -\eta\bigl(L'(w)\bigr)^2
\left(1-\frac{1}{2}\eta L''(w)\right).
$$

The first-order term decreases loss, while positive curvature produces an opposing quadratic correction. A sufficiently large learning rate can make the correction dominate.


## 2. Exact stability for a quadratic loss

Consider

$$
L(w)=\frac{a}{2}(w-w^*)^2,
\qquad a>0.
$$

Its derivative is

$$
L'(w)=a(w-w^*).
$$

Gradient descent gives

$$
w_{k+1}=w_k-\eta a(w_k-w^*).
$$

Define the error

$$
e_k=w_k-w^*.
$$

Subtract $$w^*$$ from the update:

$$
e_{k+1}=(1-\eta a)e_k.
$$

Therefore

$$
e_k=(1-\eta a)^ke_0.
$$

Convergence requires

$$
|1-\eta a|<1.
$$

Solve the double inequality:

$$
-1<1-\eta a<1.
$$

Subtract one:

$$
-2<-\eta a<0.
$$

Multiply by negative one and reverse the inequalities:

$$
0<\eta a<2.
$$

Since $$a>0,$$

$$
\boxed{0<\eta<\frac{2}{a}.}
$$

Larger curvature $$a$$ produces a smaller stable learning-rate interval.


## 3. Interpreting stable regimes

For the quadratic model:

- if $$0<\eta a<1,$$ then $$0<1-\eta a<1$$ and the error shrinks without changing sign;
- if $$\eta a=1,$$ then the exact minimum is reached in one step;
- if $$1<\eta a<2,$$ then $$-1<1-\eta a<0$$ and the error oscillates while shrinking;
- if $$\eta a=2,$$ then the multiplier is negative one and the method cycles without shrinking; and
- if $$\eta a>2,$$ then the multiplier magnitude exceeds one and the method diverges.

Overshooting is not automatically divergence. Shrinking oscillation is still convergence.


In [ ]:
a_curvature = 5.0
w_star_stability = 2.0

def quadratic_path_stability(rate, iterations=20, initial=-1.0):
    path = [initial]
    for _ in range(iterations):
        current = path[-1]
        gradient = a_curvature*(current-w_star_stability)
        path.append(current-rate*gradient)
    return np.array(path)

rates_stability = [0.05, 0.2, 0.35, 0.45]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for rate in rates_stability:
    path = quadratic_path_stability(rate)
    axes[0].plot(path, marker="o", markersize=3, label=f"learning rate = {rate}")
    axes[1].semilogy(np.maximum(np.abs(path-w_star_stability), 1e-14),
                    marker="o", markersize=3, label=f"learning rate = {rate}")
axes[0].axhline(w_star_stability, color="black", linestyle="--")
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("parameter")
axes[0].set_title("Curvature controls safe step size")
axes[1].set_xlabel("iteration")
axes[1].set_ylabel("distance from optimum")
axes[1].set_title("Stable threshold is two divided by curvature")
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


## 4. Newton's method as a curvature-aware update

The second-order local model is minimized by differentiating it with respect to the step $$h$$:

$$
\frac{d}{dh}\left(L(w)+L'(w)h+\frac{1}{2}L''(w)h^2\right)
=L'(w)+L''(w)h.
$$

Set this derivative equal to zero:

$$
L'(w)+L''(w)h=0.
$$

If $$L''(w)\ne 0,$$ then

$$
h=-\frac{L'(w)}{L''(w)}.
$$

This gives the one-variable **Newton update**

$$
\boxed{w_{k+1}=w_k-\frac{L'(w_k)}{L''(w_k)}.}
$$

Newton's method rescales the slope by curvature. It can converge quickly near a suitable minimum, but can fail when curvature is zero, negative, poorly estimated, or when the local quadratic model is unreliable.


> **Common mistakes in Section 5**
>
> - Believing the derivative alone determines a universally safe step size.
> - Calling every overshoot divergence.
> - Ignoring curvature when comparing learning rates across losses.
> - Using a local Taylor model as a global guarantee.
> - Assuming Newton's method always moves downhill.
> - Forgetting that real neural-network losses have many directions with different curvatures.

## Section 5 terminology

| Term | Meaning |
|---|---|
| Learning-rate stability | Whether repeated updates remain controlled and approach a target |
| Error multiplier | Factor mapping one iteration's distance from the optimum to the next |
| Overshoot | Update crossing to the opposite side of a minimum |
| Oscillatory convergence | Alternating errors whose magnitudes shrink |
| Curvature scale | Magnitude controlling how rapidly slope changes |
| Newton update | Step using derivative divided by second derivative |
| Local quadratic model | Second-order Taylor approximation of a loss |


## Section 5 guided worksheet

Let

$$
L(w)=4(w-1)^2.
$$

1. Rewrite the loss in the form

   $$
   \frac{a}{2}(w-w^*)^2.
   $$

2. Find the exact stability interval for the learning rate.
3. Classify the behavior for $$\eta=0.05,$$ $$0.125,$$ $$0.20,$$ and $$0.30.$$
4. Compute one Newton update from any initial value and explain the result.


In [ ]:
reveal_markdown("Open Section 5 solutions", r"""
1. Since

   $$
   4(w-1)^2=\frac{8}{2}(w-1)^2,
   $$

   we have

   $$
   a=8
   \qquad\text{and}\qquad
   w^*=1.
   $$

2. Stability requires

   $$
   0<\eta<\frac{2}{8}=\boxed{0.25}.
   $$

3. The error multiplier is

   $$
   1-8\eta.
   $$

   For $$\eta=0.05,$$ it is $$0.6,$$ giving non-oscillatory convergence.

   For $$\eta=0.125,$$ it is $$0,$$ giving exact arrival in one step.

   For $$\eta=0.20,$$ it is $$-0.6,$$ giving oscillatory convergence.

   For $$\eta=0.30,$$ it is $$-1.4,$$ giving divergence.

4. We have

   $$
   L'(w)=8(w-1)
   $$

   and

   $$
   L''(w)=8.
   $$

   Therefore

   $$
   w_{k+1}=w_k-\frac{8(w_k-1)}{8}
   =w_k-(w_k-1)
   =\boxed{1}.
   $$

   Because the loss is exactly quadratic, its second-order Taylor model is exact and Newton's method reaches the minimizer in one update.
""")


---

# Section 6: Truncation, Error, and Nonlinear AI Computation

## 1. Finite computation from an infinite expansion

Computers cannot add infinitely many terms. A Taylor series is implemented by choosing a degree and truncating:

$$
f(x)\approx T_n(x).
$$

This creates a tradeoff:

- higher degree can improve approximation;
- more terms require more arithmetic;
- large powers and factorials can create numerical issues if implemented naively; and
- approximation quality depends strongly on the input range.

Stable implementations often use range reduction, recurrences between terms, or specialized library routines rather than a direct textbook formula.


## 2. Exponential functions in sigmoid and softmax

The sigmoid uses

$$
\sigma(z)=\frac{1}{1+e^{-z}},
$$

and softmax uses

$$
\operatorname{softmax}(z_i)
=\frac{e^{z_i}}{\sum_j e^{z_j}}.
$$

Taylor expansions explain how an exponential can be approximated, but direct low-degree expansion can be poor when $$|z|$$ is large.

Softmax also requires numerical stabilization. Subtracting the largest score leaves probabilities unchanged:

$$
\frac{e^{z_i-c}}{\sum_j e^{z_j-c}}
=\frac{e^{-c}e^{z_i}}{e^{-c}\sum_j e^{z_j}}
=\frac{e^{z_i}}{\sum_j e^{z_j}}.
$$

Choosing

$$
c=\max_j z_j
$$

prevents excessively large exponentials. This is an algebraic stabilization, not a Taylor approximation.


In [ ]:
logits_softmax7 = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over="ignore", invalid="ignore"):
    naive_exp7 = np.exp(logits_softmax7)
    naive_softmax7 = naive_exp7/naive_exp7.sum()

shifted_logits7 = logits_softmax7-logits_softmax7.max()
stable_exp7 = np.exp(shifted_logits7)
stable_softmax7 = stable_exp7/stable_exp7.sum()

print("Naive exponential values:", naive_exp7)
print("Naive softmax:", naive_softmax7)
print("Shifted logits:", shifted_logits7)
print("Stable softmax:", stable_softmax7)
print("Stable probabilities sum to:", stable_softmax7.sum())


Subtracting the maximum does not alter the softmax probabilities, but it makes the largest exponent equal to $$e^0=1$$ and prevents overflow. This is a concrete example of algebraic reformulation improving numerical reliability.


In [ ]:
test_points_exp = np.array([0.1, 0.5, 1.0, 2.0, 5.0])
degrees_exp_compare = [2, 4, 8, 16]
print("Relative errors for exponential Taylor approximations")
for point in test_points_exp:
    errors = []
    for degree in degrees_exp_compare:
        approximation = float(exp_taylor(np.array([point]), degree)[0])
        errors.append(abs(approximation-np.exp(point))/np.exp(point))
    print(f"x = {point:3.1f}:", [f"{error:.3e}" for error in errors])


## 3. Error-aware approximation workflow

When replacing a nonlinear function by a polynomial:

1. identify the input range;
2. choose an expansion center near typical inputs;
3. select a degree using an error bound or validation grid;
4. evaluate numerical stability, not only symbolic error;
5. compare computational cost; and
6. test the effect on the downstream model.

A small function-approximation error can still change a classification near a decision boundary. Conversely, a larger pointwise error may be harmless in a region the model rarely visits.


> **Common mistakes in Section 6**
>
> - Assuming more terms always improve floating-point accuracy.
> - Ignoring the actual input range.
> - Evaluating factorials and powers in unnecessarily unstable ways.
> - Confusing algebraic stabilization with approximation.
> - Measuring only function error and not downstream model behavior.

## Section 6 terminology

| Term | Meaning |
|---|---|
| Truncation | Replacing an infinite expansion by finitely many terms |
| Relative error | Absolute error divided by reference magnitude |
| Range reduction | Transforming an input into a region easier to approximate |
| Numerical stability | Resistance of a computation to floating-point error |
| Stabilized softmax | Equivalent softmax computed after subtracting the largest score |
| Downstream effect | Change in the final system caused by an approximation |


## Section 6 guided worksheet

1. Use the degree-two polynomial

   $$
   T_2(x)=1+x+\frac{x^2}{2}
   $$

   to approximate $$e^{-0.2}.$$
2. Compare with the numerical value $$0.818731.$$
3. Explain why the same degree may work differently at $$x=-0.2$$ and $$x=-5.$$
4. Prove algebraically that subtracting the same constant from every softmax score does not change softmax probabilities.


In [ ]:
reveal_markdown("Open Section 6 solutions", r"""
1. Substitute $$x=-0.2$$:

   $$
   T_2(-0.2)=1-0.2+\frac{(-0.2)^2}{2}.
   $$

   Since

   $$
   (-0.2)^2=0.04,
   $$

   we obtain

   $$
   T_2(-0.2)=0.8+0.02=\boxed{0.82}.
   $$

2. The absolute error is approximately

   $$
   |0.82-0.818731|=\boxed{0.001269}.
   $$

3. Taylor error grows with distance from the center through powers of $$|x-a|.$$ The point negative five is twenty-five times farther from zero than negative zero point two, so omitted terms can be much larger.

4. For any constant $$c,$$

   $$
   \frac{e^{z_i-c}}{\sum_j e^{z_j-c}}
   =\frac{e^{-c}e^{z_i}}{e^{-c}\sum_j e^{z_j}}
   =\frac{e^{z_i}}{\sum_j e^{z_j}}.
   $$

   The common positive factor cancels from numerator and denominator.
""")


---

# Integrated Instructor Lab: Optimize a Smooth Exponential Loss

Consider

$$
L(w)=e^w-3w.
$$

The first derivative is

$$
L'(w)=e^w-3,
$$

and the second derivative is

$$
L''(w)=e^w>0.
$$

Because the curvature is positive everywhere, any critical point is the unique global minimum. Solve

$$
e^w-3=0.
$$

Then

$$
e^w=3
$$

and

$$
\boxed{w^*=\ln 3\approx 1.098612.}
$$

### A second-order approximation centered at one

Write

$$
w=1+h.
$$

Then

$$
e^w=e^{1+h}=e\,e^h.
$$

Using

$$
e^h\approx 1+h+\frac{h^2}{2},
$$

gives

$$
e^w\approx e\left(1+(w-1)+\frac{(w-1)^2}{2}\right).
$$

This local quadratic supplies an approximation to the loss near $$w=1.$$


## Gradient descent and Newton updates

Gradient descent with learning rate $$\eta$$ is

$$
w_{k+1}=w_k-\eta(e^{w_k}-3).
$$

Newton's method is

$$
w_{k+1}=w_k-\frac{e^{w_k}-3}{e^{w_k}}.
$$

Simplifying,

$$
w_{k+1}=w_k-1+3e^{-w_k}.
$$

These generate two different sequences. Newton uses curvature and often moves rapidly near the solution, while gradient descent uses a fixed scale and may be more gradual.


In [ ]:
def loss_lab7(w):
    return np.exp(w)-3*w

def gradient_lab7(w):
    return np.exp(w)-3

def newton_path_lab7(initial, iterations):
    path = [initial]
    for _ in range(iterations):
        current = path[-1]
        path.append(current-gradient_lab7(current)/np.exp(current))
    return np.array(path)

def gradient_path_lab7(initial, learning_rate, iterations):
    path = [initial]
    for _ in range(iterations):
        current = path[-1]
        path.append(current-learning_rate*gradient_lab7(current))
    return np.array(path)

w_star_lab7 = np.log(3)
path_newton_lab7 = newton_path_lab7(0.0, 7)
path_gradient_lab7 = gradient_path_lab7(0.0, 0.1, 30)

w_grid_lab7 = np.linspace(-0.5, 2.5, 500)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(w_grid_lab7, loss_lab7(w_grid_lab7), linewidth=2)
axes[0].scatter(path_gradient_lab7, loss_lab7(path_gradient_lab7), s=22,
                label="gradient descent")
axes[0].scatter(path_newton_lab7, loss_lab7(path_newton_lab7), s=45,
                marker="x", label="Newton")
axes[0].set_xlabel("parameter w")
axes[0].set_ylabel("loss")
axes[0].set_title("Two optimization sequences")
axes[0].legend()
axes[1].semilogy(np.maximum(np.abs(path_gradient_lab7-w_star_lab7), 1e-15),
                  marker="o", markersize=3, label="gradient descent")
axes[1].semilogy(np.maximum(np.abs(path_newton_lab7-w_star_lab7), 1e-15),
                  marker="x", label="Newton")
axes[1].set_xlabel("iteration")
axes[1].set_ylabel("distance from exact minimizer")
axes[1].set_title("Convergence comparison")
axes[1].legend()
plt.tight_layout()
plt.show()

print("Exact minimizer:", w_star_lab7)
print("Newton sequence:", path_newton_lab7)


## Integrated lab tasks

1. Compute the first two gradient-descent updates from $$w_0=0$$ with $$\eta=0.1.$$
2. Compute the first two Newton updates from $$w_0=0.$$
3. Explain why positive second derivative guarantees the critical point is a minimum.
4. Construct the second-order Taylor approximation of the entire loss centered at $$w=1.$$
5. Compare the meanings of sequence convergence, Taylor-series convergence, and optimization convergence.


In [ ]:
reveal_markdown("Open integrated-lab solution", r"""
### Gradient descent

At $$w_0=0,$$

$$
L'(0)=1-3=-2.
$$

Therefore

$$
w_1=0-0.1(-2)=0.2.
$$

Next,

$$
L'(0.2)=e^{0.2}-3
\approx 1.221403-3
=-1.778597.
$$

Thus

$$
w_2=0.2-0.1(-1.778597)
\approx \boxed{0.377860}.
$$

### Newton's method

At $$w_0=0,$$

$$
w_1=0-\frac{1-3}{1}=2.
$$

At $$w_1=2,$$

$$
w_2=2-\frac{e^2-3}{e^2}
=2-1+\frac{3}{e^2}
\approx \boxed{1.406006}.
$$

### Taylor approximation of the loss

At the center $$w=1,$$

$$
L(1)=e-3,
$$

$$
L'(1)=e-3,
$$

and

$$
L''(1)=e.
$$

Therefore

$$
\boxed{T_2(w)
=(e-3)+(e-3)(w-1)+\frac{e}{2}(w-1)^2.}
$$

Positive curvature everywhere means the slope increases continuously and the stationary point is the unique global minimum.

Sequence convergence concerns indexed values approaching a limit. Taylor-series convergence concerns partial polynomial sums approaching a function value. Optimization convergence concerns an algorithm's parameter or loss sequence approaching an optimum or stationary point.
""")


---

# Module 7 Summary

1. A sequence converges when all sufficiently late terms approach one finite limit.
2. A series converges when its sequence of partial sums converges.
3. Terms approaching zero are necessary but not sufficient for series convergence.
4. Geometric series connect directly to discounted rewards and iterative error recurrences.
5. Taylor polynomials match derivative information at a center.
6. First order captures slope; second order adds curvature.
7. Approximation reliability depends on degree, distance, smoothness, and numerical implementation.
8. For a quadratic loss, learning-rate stability is determined exactly by curvature.
9. Newton's method uses a local quadratic model to rescale the gradient.

## Cumulative terminology

| AI language | Sequence, series, or Taylor interpretation |
|---|---|
| Parameter trajectory | Recursively generated sequence |
| Stable training | Update errors shrink rather than grow |
| Discounted return | Geometric series of future rewards |
| Local loss approximation | Taylor polynomial near current parameters |
| Learning-rate threshold | Bound determined by local curvature in a quadratic model |
| Overshooting | Crossing an optimum because of a large update |
| Approximate activation | Truncated expansion or other finite numerical method |

## Exit ticket

1. Why does $$a_n\to 0$$ not prove that a series converges?
2. What object must converge for an infinite series to have a sum?
3. What information is added by a second-order Taylor term?
4. Why does high curvature require a smaller stable learning rate?
5. Why can a polynomial approximation that is excellent near its center fail far away?
